# Klasifikasi Tutupan Lahan (SVM)
Alur Kerja Script: ekstrak sampel -> Hitung separability (JM Distance) -> Split data training 70% dan validasi 30% -> Generate algoritma SVM -> Uji Validasi (Kappa/OA/PA/UA -> Output Excel) -> klasifikasi Tutupan Lahan Tahun 2020 & 2023 (Output .tif)

In [ ]:
import os, glob, itertools, json
from concurrent.futures import ThreadPoolExecutor
import xml.etree.ElementTree as ET
import struct
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import mask as rio_mask
from rasterio.windows import Window
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import confusion_matrix, cohen_kappa_score, accuracy_score

# ================= KONFIGURASI =================
SHP_TRAINING = r"D:\AUZAIE\LAMAR KERJA\LAMAR GAWEAN\LAMAR KERJAAN\LAMAR GIS ANALYST ILLAPS\BAHAN TES\Training_Sample_Separate.shp"
FIELD_KELAS  = "Classname"
DIR_CITRA    = r"D:\AUZAIE\LAMAR KERJA\LAMAR GAWEAN\LAMAR KERJAAN\LAMAR GIS ANALYST ILLAPS\BAHAN TES\BAHAN\Citra"
DIR_OUTPUT   = r"D:\AUZAIE\LAMAR KERJA\LAMAR GAWEAN\LAMAR KERJAAN\LAMAR GIS ANALYST ILLAPS\BAHAN TES\BAHAN\output"
os.makedirs(DIR_OUTPUT, exist_ok=True)

def cari_citra(tahun):
    hasil = glob.glob(os.path.join(DIR_CITRA, f"*{tahun}*.tif"))
    if not hasil:
        raise FileNotFoundError(f"Citra tahun {tahun} tidak ditemukan di {DIR_CITRA}")
    return hasil[0]

CITRA_2020 = cari_citra(2020)
CITRA_2023 = cari_citra(2023)
CITRA_REFERENSI = CITRA_2020   

TEST_SIZE    = 0.3  
RANDOM_STATE = 42
BLOCK_SIZE   = 512   
MAX_SAMPEL_PER_KELAS = 3000  

print("Citra 2020:", CITRA_2020)
print("Citra 2023:", CITRA_2023)

## 1. Ekstraksi Sampel Piksel Training
Menggunakan dasar perhitungan jumlah pixel dalam poligon data training

In [ ]:
def ekstrak_sampel(shp_path, raster_path, field_kelas):
    gdf = gpd.read_file(shp_path)
    with rasterio.open(raster_path) as src:
        if gdf.crs != src.crs:
            gdf = gdf.to_crs(src.crs)
        X_list, y_list = [], []
        for _, row in gdf.iterrows():
            try:
                out_img, _ = rio_mask.mask(src, [row.geometry], crop=True, filled=False)
            except ValueError:
                continue
            out_img = np.ma.filled(out_img.astype(np.float32), np.nan)
            bands, h, w = out_img.shape
            pix = out_img.reshape(bands, -1).T
            pix = pix[~np.isnan(pix).any(axis=1)]
            if pix.size == 0:
                continue
            X_list.append(pix)
            y_list.append(np.full(pix.shape[0], row[field_kelas]))
    X = np.vstack(X_list)
    y = np.concatenate(y_list)
    print(f"Total sampel piksel: {X.shape[0]} | Jumlah band: {X.shape[1]} | Jumlah kelas: {len(np.unique(y))}")
    return X, y

## 2. Separability Index — Metode Jeffries-Matusita (JM) Distance
Digunakan untuk menghitung index keterpisahan data training yang mewakili setiap objek
Kategori Index -> 1.90-2.00: Sangat Baik | 1.00-1.89: Sedang | 0.00-0.99: Buruk

In [ ]:
def hitung_jm_distance(X, y):
    kelas = np.unique(y)
    n = len(kelas)
    mat = pd.DataFrame(np.zeros((n, n)), index=kelas, columns=kelas)

    stat = {}
    for k in kelas:
        Xk = X[y == k]
        cov = np.cov(Xk, rowvar=False) + np.eye(Xk.shape[1]) * 1e-6
        stat[k] = (np.mean(Xk, axis=0), cov)

    def keterangan(v):
        if v >= 1.9:
            return "Sangat Baik"
        elif v >= 1.0:
            return "Sedang"
        else:
            return "Buruk"

    detail = []
    for a, b in itertools.combinations(kelas, 2):
        mu1, cov1 = stat[a]
        mu2, cov2 = stat[b]
        cov_avg = (cov1 + cov2) / 2
        diff = (mu1 - mu2).reshape(-1, 1)
        inv_cov = np.linalg.pinv(cov_avg)
        term1 = float(0.125 * (diff.T @ inv_cov @ diff)[0, 0])

        det1 = max(np.linalg.det(cov1), 1e-300)
        det2 = max(np.linalg.det(cov2), 1e-300)
        det_avg = max(np.linalg.det(cov_avg), 1e-300)
        term2 = 0.5 * np.log(det_avg / np.sqrt(det1 * det2))

        B = term1 + term2
        JM = 2 * (1 - np.exp(-B))
        mat.loc[a, b] = mat.loc[b, a] = JM
        detail.append((a, b, JM, keterangan(JM)))

    print("=== Matriks JM Distance ===")
    print(mat.round(3))

    print("\n=== Keterangan Separabilitas Antar Pasangan Kelas ===")
    lebar = max(len(str(k)) for k in kelas)
    header = f"{'Kelas A':<{lebar}}  {'Kelas B':<{lebar}}  {'JM':>5}  Keterangan"
    print(header)
    print("-" * len(header))
    for a, b, jm, ket in detail:
        print(f"{a:<{lebar}}  {b:<{lebar}}  {jm:>5.3f}  {ket}")
    print("\nSkala -> 1.90-2.00: Sangat Baik | 1.00-1.89: Sedang | 0.00-0.99: Buruk")

    return mat, detail

## 3. Split Data Training (70%) dan Validasi (30%) — Metode Random, Stratified

In [ ]:
def split_data(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    total = X_train.shape[0] + X_val.shape[0]
    print(f"Training : {X_train.shape[0]} sampel ({X_train.shape[0]/total:.0%}) | Validasi : {X_val.shape[0]} sampel ({X_val.shape[0]/total:.0%}) [target validasi = {test_size:.0%}]")
    return X_train, X_val, y_train, y_val

## 4. Training Klasifikasi Support Vector Machine (SVM)

In [ ]:
def latih_svm(X_train, y_train, max_per_class=MAX_SAMPEL_PER_KELAS):
    # Subsampling stratified per kelas agar training SVM lebih cepat (tidak mengubah data validasi)
    if max_per_class is not None:
        idx_final = []
        rng = np.random.RandomState(RANDOM_STATE)
        for k in np.unique(y_train):
            idx_k = np.where(y_train == k)[0]
            if len(idx_k) > max_per_class:
                idx_k = rng.choice(idx_k, size=max_per_class, replace=False)
            idx_final.append(idx_k)
        idx_final = np.concatenate(idx_final)
        X_train, y_train = X_train[idx_final], y_train[idx_final]
        print(f"Sampel training setelah subsampling: {X_train.shape[0]} (maks {max_per_class}/kelas)")

    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    model = SVC(kernel="rbf", C=100, gamma="scale", cache_size=2000)
    model.fit(X_train_s, y_train)
    print("Training SVM selesai.")
    return model, scaler

## 5. Validasi Model — Cohen's Kappa, Overall Accuracy, Producer's & User's Accuracy -> Output Excel

In [ ]:
def validasi_model(model, scaler, X_val, y_val, label_encoder, output_excel):
    X_val_s = scaler.transform(X_val)
    y_pred = model.predict(X_val_s)

    nama_kelas = label_encoder.classes_
    n = len(nama_kelas)
    cm = confusion_matrix(y_val, y_pred, labels=range(n))

    oa = accuracy_score(y_val, y_pred)
    kappa = cohen_kappa_score(y_val, y_pred)

    total_aktual   = cm.sum(axis=1)
    total_prediksi = cm.sum(axis=0)
    pa = np.divide(np.diag(cm), total_aktual, out=np.zeros(n), where=total_aktual != 0)
    ua = np.divide(np.diag(cm), total_prediksi, out=np.zeros(n), where=total_prediksi != 0)

    df_cm = pd.DataFrame(cm, index=nama_kelas, columns=nama_kelas)
    df_acc = pd.DataFrame({
        "Kelas": nama_kelas,
        "Producer's Accuracy (%)": np.round(pa * 100, 2),
        "User's Accuracy (%)": np.round(ua * 100, 2),
    })
    df_summary = pd.DataFrame({
        "Metrik": ["Overall Accuracy (%)", "Cohen's Kappa"],
        "Nilai": [round(oa * 100, 2), round(kappa, 4)],
    })

    with pd.ExcelWriter(output_excel, engine="openpyxl") as writer:
        df_summary.to_excel(writer, sheet_name="Ringkasan", index=False)
        df_acc.to_excel(writer, sheet_name="PA_UA", index=False)
        df_cm.to_excel(writer, sheet_name="Confusion_Matrix")

    print(f"Overall Accuracy : {oa*100:.2f}%")
    print(f"Cohen's Kappa    : {kappa:.4f}")
    print(f"Hasil validasi disimpan di: {output_excel}")

## 6. Klasifikasi Citra -> Output .tif

In [ ]:
def tulis_pam_aux_xml(tif_path, nama_kelas):
    root = ET.Element("PAMDataset")
    band = ET.SubElement(root, "PAMRasterBand", band="1")
    ET.SubElement(band, "Description").text = "Tutupan Lahan (Value = Kode Kelas)"

    cat_names = ET.SubElement(band, "CategoryNames")
    ET.SubElement(cat_names, "Category").text = "Tidak Terklasifikasi"
    for nm in nama_kelas:
        ET.SubElement(cat_names, "Category").text = str(nm)

    rat = ET.SubElement(band, "GDALRasterAttributeTable")
    f0 = ET.SubElement(rat, "FieldDefn", index="0")
    ET.SubElement(f0, "Name").text = "Value"
    ET.SubElement(f0, "Type").text = "0"
    ET.SubElement(f0, "Usage").text = "0"
    f1 = ET.SubElement(rat, "FieldDefn", index="1")
    ET.SubElement(f1, "Name").text = "Classname"
    ET.SubElement(f1, "Type").text = "2"
    ET.SubElement(f1, "Usage").text = "2"

    row0 = ET.SubElement(rat, "Row", index="0")
    ET.SubElement(row0, "F").text = "0"
    ET.SubElement(row0, "F").text = "Tidak Terklasifikasi"
    for i, nm in enumerate(nama_kelas, start=1):
        row = ET.SubElement(rat, "Row", index=str(i))
        ET.SubElement(row, "F").text = str(i)
        ET.SubElement(row, "F").text = str(nm)

    aux_path = tif_path + ".aux.xml"
    ET.ElementTree(root).write(aux_path, encoding="UTF-8", xml_declaration=True)
    print("Atribut kelas (RAT/PAM, tanpa osgeo) ditulis ke:", aux_path)


def tulis_legenda_csv(output_dir, nama_kelas):
    mapping = {i + 1: str(c) for i, c in enumerate(nama_kelas)}
    legenda_path = os.path.join(output_dir, "Legenda_Kelas.csv")
    pd.DataFrame({"Kode": list(mapping.keys()), "Classname": list(mapping.values())}).to_csv(legenda_path, index=False)
    print("Legenda kelas disimpan di:", legenda_path)


def tulis_vat_dbf(tif_path, nama_kelas, block_size=BLOCK_SIZE):
    n_kelas = len(nama_kelas)
    counts = np.zeros(n_kelas + 1, dtype=np.int64)
    with rasterio.open(tif_path) as src:
        for i in range(0, src.height, block_size):
            for j in range(0, src.width, block_size):
                h = min(block_size, src.height - i)
                w = min(block_size, src.width - j)
                block = src.read(1, window=Window(j, i, w, h))
                counts += np.bincount(block.ravel(), minlength=n_kelas + 1)[:n_kelas + 1]

    field_defs = [(b"VALUE", b"N", 10, 0), (b"COUNT", b"N", 10, 0), (b"Classname", b"C", 60, 0)]
    rec_len = 1 + sum(f[2] for f in field_defs)
    header_len = 32 + 32 * len(field_defs) + 1
    dbf_path = tif_path + ".vat.dbf"

    with open(dbf_path, "wb") as f:
        f.write(bytes([0x03]))
        f.write(bytes([100, 1, 1]))
        f.write(struct.pack("<I", n_kelas))
        f.write(struct.pack("<H", header_len))
        f.write(struct.pack("<H", rec_len))
        f.write(b"\x00" * 20)
        for name, ftype, flen, dec in field_defs:
            f.write(name[:10].ljust(11, b"\x00"))
            f.write(ftype)
            f.write(b"\x00" * 4)
            f.write(bytes([flen]))
            f.write(bytes([dec]))
            f.write(b"\x00" * 14)
        f.write(b"\x0D")
        for idx in range(n_kelas):
            f.write(b" ")
            f.write(str(idx + 1).rjust(10)[:10].encode("ascii"))
            f.write(str(int(counts[idx + 1])).rjust(10)[:10].encode("ascii"))
            f.write(str(nama_kelas[idx]).ljust(60)[:60].encode("ascii", errors="replace"))
        f.write(b"\x1A")
    print("VAT (Value Attribute Table) untuk ArcGIS ditulis ke:", dbf_path)


def tulis_kelas_ke_tif(tif_path, nama_kelas):
    palet = [(31,120,180),(51,160,44),(227,26,28),(255,127,0),(106,61,154),
             (166,206,227),(178,223,138),(251,154,153),(253,191,111),(202,178,214)]
    mapping = {i + 1: str(c) for i, c in enumerate(nama_kelas)}

    try:
        with rasterio.open(tif_path, "r+") as dst:
            colormap = {0: (0, 0, 0, 0)}
            for kode, nm in mapping.items():
                r, g, b = palet[(kode - 1) % len(palet)]
                colormap[kode] = (r, g, b, 255)
            dst.write_colormap(1, colormap)
            dst.update_tags(1, CLASS_MAP=json.dumps(mapping))
            dst.set_band_description(1, "Tutupan Lahan (lihat tag CLASS_MAP / file legenda)")
        print("Colormap + tag CLASS_MAP ditulis ke raster:", mapping)
    except Exception as e:
        print("Peringatan: gagal menulis colormap/tag ke raster ->", e)

    tulis_pam_aux_xml(tif_path, nama_kelas)
    tulis_vat_dbf(tif_path, nama_kelas)


def klasifikasi_citra(raster_path, model, scaler, label_encoder, output_path,
                       block_size=BLOCK_SIZE, n_workers=None):
    n_workers = n_workers or (os.cpu_count() or 4)

    with rasterio.open(raster_path) as src:
        profile = src.profile.copy()
        profile.update(count=1, dtype="uint8", compress="lzw", nodata=0)
        nodata_src = src.nodata
        height, width = src.height, src.width

    windows = [
        Window(j, i, min(block_size, width - j), min(block_size, height - i))
        for i in range(0, height, block_size)
        for j in range(0, width, block_size)
    ]

    def proses_block(win):
        with rasterio.open(raster_path) as src_local:  # handle terpisah per-thread
            block = src_local.read(window=win).astype(np.float32)
        bands, hh, ww = block.shape
        flat = block.reshape(bands, -1).T

        if nodata_src is not None:
            valid = ~np.any(flat == nodata_src, axis=1)
        else:
            valid = ~np.isnan(flat).any(axis=1)

        out_flat = np.zeros(flat.shape[0], dtype=np.uint8)
        if valid.any():
            pred = model.predict(scaler.transform(flat[valid]))
            out_flat[valid] = pred + 1
        return win, out_flat.reshape(hh, ww)

    with rasterio.open(output_path, "w", **profile) as dst:
        with ThreadPoolExecutor(max_workers=n_workers) as ex:
            for win, out_block in ex.map(proses_block, windows):
                dst.write(out_block, 1, window=win)  # tulis tetap sekuensial (aman)

    tulis_kelas_ke_tif(output_path, label_encoder.classes_)
    print(f"Klasifikasi selesai -> {output_path}")

## 7. Eksekusi — Ekstraksi Sampel & JM Distance

In [ ]:
X, y = ekstrak_sampel(SHP_TRAINING, CITRA_REFERENSI, FIELD_KELAS)
mat_jm, detail_jm = hitung_jm_distance(X, y)

## 8. Eksekusi — Split, Training, Validasi

In [ ]:
# ===== Tahun 2020: ekstraksi sampel, split, training, validasi =====
X_2020, y_2020 = ekstrak_sampel(SHP_TRAINING, CITRA_2020, FIELD_KELAS)
le_2020 = LabelEncoder()
y_2020_enc = le_2020.fit_transform(y_2020)
X_train_2020, X_val_2020, y_train_2020, y_val_2020 = split_data(X_2020, y_2020_enc)
model_2020, scaler_2020 = latih_svm(X_train_2020, y_train_2020)
validasi_model(model_2020, scaler_2020, X_val_2020, y_val_2020, le_2020,
                os.path.join(DIR_OUTPUT, "Hasil Validasi Tutupan Lahan Tahun 2020.xlsx"))

In [ ]:
# ===== Tahun 2023: ekstraksi sampel, split, training, validasi =====
X_2023, y_2023 = ekstrak_sampel(SHP_TRAINING, CITRA_2023, FIELD_KELAS)
le_2023 = LabelEncoder()
y_2023_enc = le_2023.fit_transform(y_2023)
X_train_2023, X_val_2023, y_train_2023, y_val_2023 = split_data(X_2023, y_2023_enc)
model_2023, scaler_2023 = latih_svm(X_train_2023, y_train_2023)
validasi_model(model_2023, scaler_2023, X_val_2023, y_val_2023, le_2023,
                os.path.join(DIR_OUTPUT, "Hasil Validasi Tutupan Lahan Tahun 2023.xlsx"))

## 9. Eksekusi — Klasifikasi Citra 2020 & 2023

In [ ]:
klasifikasi_citra(CITRA_2020, model_2020, scaler_2020, le_2020,
                   os.path.join(DIR_OUTPUT, "Tutupan_Lahan_2020.tif"))

In [ ]:
klasifikasi_citra(CITRA_2023, model_2023, scaler_2023, le_2023,
                   os.path.join(DIR_OUTPUT, "Tutupan_Lahan_2023.tif"))

tulis_legenda_csv(DIR_OUTPUT, le_2023.classes_)
print("\nSELESAI. Output tersimpan di:", DIR_OUTPUT)